# DepthWizard -- Land-cover Segmentation Fine-tuning

Fine-tunes the DeepLabV3 (MobileNetV3-Large backbone, 7-class head) landcover
segmentation model used by `backend/app/models/landcover_model.py`. This
model feeds Stage 2's per-class height calibration AND Stage 3's building
footprint / roof-plane / tree-instance post-processing -- so segmentation
quality here has downstream effect on almost everything judges see.

**Read `AGENT_FINETUNE_GUIDE.md` in this same folder before you start** --
it explains the run -> screenshot -> share-with-agent -> update loop this
notebook is meant to be used with.

Classes (fixed, must match `landcover_model.py` exactly):
`0=unlabeled 1=ground 2=low_vegetation 3=building 4=water 5=road 6=tree`

**Runtime:** Runtime -> Change runtime type -> GPU (T4 is fine) before running anything.


## 1. Setup

In [ ]:
!pip install -q huggingface_hub h5py torchmetrics


In [ ]:
import io
import os
import json
import time
import random
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected -- Runtime > Change runtime type > GPU")


## 2. Config

Edit these. `RUN_NAME` matters most if you're one of two people fine-tuning
in parallel -- give each approach a distinct name so checkpoints/exports
never collide or get overwritten by the other person's run.


In [ ]:
# ---- identity for this run (IMPORTANT if 2 people run this in parallel) ----
RUN_NAME = "teamA_v1"          # e.g. "teamA_v1", "teamB_v1" -- change per person/attempt

# ---- data ----
NUM_TRAIN_SAMPLES = 600         # GAMUS samples to cache locally for training
NUM_VAL_SAMPLES = 100
GAMUS_SPLIT_TRAIN = "train"
GAMUS_SPLIT_VAL = "test"
CACHE_DIR = Path(f"/content/gamus_cache_{RUN_NAME}")

# ---- model ----
IMG_SIZE = 512                  # must match landcover_model.MODEL_INPUT_SIZE -- do not change
NUM_CLASSES = 7
CLASS_NAMES = ["unlabeled", "ground", "low_vegetation", "building", "water", "road", "tree"]

# ---- starting point: continue from the existing checkpoint (recommended) ----
# Upload the current best checkpoint (e.g. landcover_seg_v6.pth) via the file
# browser on the left, or mount Drive below, then set this path. Leave as
# None to train a fresh backbone from torchvision ImageNet weights instead
# (only do that if you specifically want to try a different starting point).
RESUME_FROM_CHECKPOINT = None    # e.g. "/content/landcover_seg_v6.pth"

# ---- training hyperparameters -- THESE are what you'll mostly be tuning ----
EPOCHS = 15
BATCH_SIZE = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
FREEZE_BACKBONE_EPOCHS = 0       # >0 freezes backbone for the first N epochs (fine-tune head only first)
USE_CLASS_WEIGHTS = True         # weight loss by inverse class frequency -- GAMUS is class-imbalanced
AUGMENT = True

# ---- output ----
OUTPUT_DIR = Path(f"/content/checkpoints_{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_NAME = f"landcover_seg_{RUN_NAME}.pth"   # distinct filename per run -- do not rename to landcover_seg_v6.pth until you've actually decided to replace it

random.seed(0); np.random.seed(0); torch.manual_seed(0)
print(f"Run: {RUN_NAME}  ->  will save to {OUTPUT_DIR / CHECKPOINT_NAME}")


## 3. Dataset -- stream GAMUS from Hugging Face, cache locally

Streaming every epoch from HF is slow and wastes your Colab session time on
network I/O instead of training. This downloads a fixed number of samples
ONCE into local `.npy` files, then the `Dataset` just reads from disk for
every epoch after that.

Uses the exact same field-name-guessing logic as
`backend/app/data/gamus_loader.py` (kept in sync deliberately) so this
notebook keeps working if GAMUS's HF schema is confirmed/changed later --
if it raises a `KeyError` listing available keys, update
`_FIELD_CANDIDATES` in BOTH this cell and `gamus_loader.py`.


In [ ]:
_FIELD_CANDIDATES = {
    "rgb": ["image", "rgb", "rgb_image", "img"],
    "landcover": ["landcover", "land_cover", "semantic", "mask", "class_mask"],
}

def _stream_h5_array(repo_filepath: str):
    url = f"https://huggingface.co/datasets/earthflow/GAMUS/resolve/main/{repo_filepath}"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        data = resp.read()
    import h5py
    with h5py.File(io.BytesIO(data), "r") as f:
        key = list(f.keys())[0]
        return f[key][:]

def cache_gamus_split(split: str, n_samples: int, cache_dir: Path):
    from huggingface_hub import HfApi
    cache_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = cache_dir / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if len(manifest) >= n_samples:
            print(f"[{split}] cache already has {len(manifest)} samples, reusing.")
            return manifest[:n_samples]

    api = HfApi()
    all_files = list(api.list_repo_files(repo_id="earthflow/GAMUS", repo_type="dataset"))
    prefix = f"images/{split}/"
    rgb_files = [f for f in all_files if f.startswith(prefix) and f.endswith("_RGB.h5")]
    random.Random(0).shuffle(rgb_files)
    rgb_files = rgb_files[:n_samples]

    manifest = []
    for i, rgb_rel in enumerate(rgb_files):
        base_name = rgb_rel[len(prefix):-len("_RGB.h5")]
        cls_rel = f"classes/{split}/{base_name}_CLS.h5"
        if cls_rel not in all_files:
            continue
        try:
            rgb = _stream_h5_array(rgb_rel)
            cls_mask = _stream_h5_array(cls_rel)
        except Exception as e:
            print(f"  skip {base_name}: {e}")
            continue
        np.save(cache_dir / f"{i:05d}_rgb.npy", rgb)
        np.save(cache_dir / f"{i:05d}_cls.npy", cls_mask.astype(np.int64))
        manifest.append(f"{i:05d}")
        if (i + 1) % 25 == 0:
            print(f"[{split}] cached {i+1}/{len(rgb_files)}")
    manifest_path.write_text(json.dumps(manifest))
    print(f"[{split}] done -- {len(manifest)} samples cached to {cache_dir}")
    return manifest

train_cache_dir = CACHE_DIR / "train"
val_cache_dir = CACHE_DIR / "val"
train_manifest = cache_gamus_split(GAMUS_SPLIT_TRAIN, NUM_TRAIN_SAMPLES, train_cache_dir)
val_manifest = cache_gamus_split(GAMUS_SPLIT_VAL, NUM_VAL_SAMPLES, val_cache_dir)


In [ ]:
class GAMUSSegDataset(Dataset):
    """Reads cached RGB + class-mask .npy pairs, center-crops/resizes to
    IMG_SIZE (matching normalize_gsd()'s behavior at inference time -- keep
    these consistent or the model trains on a distribution it won't see in
    production), and applies light augmentation for the training split."""

    def __init__(self, cache_dir: Path, manifest, augment: bool = False):
        self.cache_dir = cache_dir
        self.manifest = manifest
        self.augment = augment

    def __len__(self):
        return len(self.manifest)

    def _center_crop_resize(self, rgb, mask):
        h, w = rgb.shape[:2]
        side = min(h, w)
        top, left = (h - side) // 2, (w - side) // 2
        rgb = rgb[top:top + side, left:left + side]
        mask = mask[top:top + side, left:left + side]
        rgb_t = torch.from_numpy(rgb).permute(2, 0, 1).float().unsqueeze(0)
        mask_t = torch.from_numpy(mask).float().unsqueeze(0).unsqueeze(0)
        rgb_t = F.interpolate(rgb_t, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        mask_t = F.interpolate(mask_t, size=(IMG_SIZE, IMG_SIZE), mode="nearest")
        return rgb_t.squeeze(0), mask_t.squeeze(0).squeeze(0).long()

    def __getitem__(self, idx):
        stem = self.manifest[idx]
        rgb = np.load(self.cache_dir / f"{stem}_rgb.npy").astype(np.float32)
        mask = np.load(self.cache_dir / f"{stem}_cls.npy")
        if rgb.ndim == 2:
            rgb = np.stack([rgb] * 3, axis=-1)
        rgb_t, mask_t = self._center_crop_resize(rgb, mask)

        if self.augment:
            if random.random() < 0.5:
                rgb_t = torch.flip(rgb_t, dims=[2])
                mask_t = torch.flip(mask_t, dims=[1])
            if random.random() < 0.5:
                rgb_t = torch.flip(rgb_t, dims=[1])
                mask_t = torch.flip(mask_t, dims=[0])
            if random.random() < 0.3:
                factor = 0.8 + random.random() * 0.4
                rgb_t = (rgb_t * factor).clamp(0, 255)

        rgb_t = rgb_t / 255.0
        return rgb_t, mask_t


train_dataset = GAMUSSegDataset(train_cache_dir, train_manifest, augment=AUGMENT)
val_dataset = GAMUSSegDataset(val_cache_dir, val_manifest, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
rgb_sample, mask_sample = train_dataset[0]
print("Sample shapes:", rgb_sample.shape, mask_sample.shape, "mask classes present:", torch.unique(mask_sample).tolist())


## 4. Class weights (optional, `USE_CLASS_WEIGHTS`)

GAMUS is heavily skewed toward ground/road/low-vegetation and light on water/building in many tiles -- unweighted CE lets the model ignore rare classes. Computed once from the cached training set.

In [ ]:
def compute_class_weights(dataset, num_classes):
    counts = torch.zeros(num_classes)
    for i in range(len(dataset)):
        _, mask = dataset[i]
        for c in range(num_classes):
            counts[c] += (mask == c).sum()
    counts = counts.clamp(min=1)
    weights = counts.sum() / (num_classes * counts)
    return weights

if USE_CLASS_WEIGHTS:
    print("Computing class weights from training cache (one pass, cached data so this is fast)...")
    class_weights = compute_class_weights(train_dataset, NUM_CLASSES).to(device)
    for name, w in zip(CLASS_NAMES, class_weights.tolist()):
        print(f"  {name:16s} weight={w:.3f}")
else:
    class_weights = None


## 5. Model -- must match `landcover_model.py` exactly

Same architecture: `deeplabv3_mobilenet_v3_large`, classifier head replaced for `NUM_CLASSES`. If you change the architecture here, the resulting checkpoint will NOT load in `load_landcover_model()` without also updating that file to match -- don't change this cell casually.

In [ ]:
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large

def build_model(num_classes, pretrained_backbone=True):
    model = deeplabv3_mobilenet_v3_large(
        weights=None,
        weights_backbone="IMAGENET1K_V1" if pretrained_backbone else None,
        aux_loss=True,   # aux head helps training signal; inference-side loader drops these weights, see landcover_model.py
    )
    model.classifier[-1] = nn.Conv2d(256, num_classes, kernel_size=1)
    if model.aux_classifier is not None:
        model.aux_classifier[-1] = nn.Conv2d(10, num_classes, kernel_size=1)
    return model

model = build_model(NUM_CLASSES, pretrained_backbone=(RESUME_FROM_CHECKPOINT is None))

start_epoch = 0
best_iou_so_far = 0.0
if RESUME_FROM_CHECKPOINT is not None:
    ckpt = torch.load(RESUME_FROM_CHECKPOINT, map_location=device)
    state_dict = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Resumed from {RESUME_FROM_CHECKPOINT} -- missing={len(missing)} unexpected={len(unexpected)}")
    if isinstance(ckpt, dict):
        best_iou_so_far = ckpt.get("best_iou", 0.0)
        print(f"Checkpoint's recorded best_iou: {best_iou_so_far:.4f} (this run will only overwrite the saved file if it beats this)")

model = model.to(device)
print("Model ready.")


## 6. Metrics -- per-class + mean IoU

This is the table to screenshot for your agent after each run (see `AGENT_FINETUNE_GUIDE.md`) -- it's the actual signal for what to change next (e.g. "building IoU is low" -> more building-heavy augmentation or a higher building class weight).

In [ ]:
@torch.no_grad()
def evaluate(model, loader, num_classes):
    model.eval()
    intersection = torch.zeros(num_classes)
    union = torch.zeros(num_classes)
    for rgb, mask in loader:
        rgb, mask = rgb.to(device), mask.to(device)
        pred = model(rgb)["out"].argmax(1)
        for c in range(num_classes):
            pred_c = (pred == c)
            mask_c = (mask == c)
            intersection[c] += (pred_c & mask_c).sum().item()
            union[c] += (pred_c | mask_c).sum().item()
    iou_per_class = intersection / union.clamp(min=1)
    present = union > 0
    mean_iou = iou_per_class[present].mean().item() if present.any() else 0.0
    return mean_iou, iou_per_class


def print_iou_table(iou_per_class, epoch=None, mean_iou=None):
    header = f"--- IoU per class" + (f" (epoch {epoch})" if epoch is not None else "") + " ---"
    print(header)
    for name, iou in zip(CLASS_NAMES, iou_per_class.tolist()):
        print(f"  {name:16s} {iou:.4f}")
    if mean_iou is not None:
        print(f"  {'MEAN':16s} {mean_iou:.4f}")


## 7. Train loop

Saves the checkpoint (in the SAME format `landcover_model.py` expects: `{"model_state", "optimizer_state", "epoch", "best_iou"}`) only when validation mean IoU improves -- so `checkpoints_<RUN_NAME>/<CHECKPOINT_NAME>` always holds this run's best, not just its last epoch.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "val_mean_iou": []}
best_iou = best_iou_so_far

for epoch in range(EPOCHS):
    if FREEZE_BACKBONE_EPOCHS > 0:
        for p in model.backbone.parameters():
            p.requires_grad = epoch >= FREEZE_BACKBONE_EPOCHS

    model.train()
    epoch_loss = 0.0
    t0 = time.time()
    for rgb, mask in train_loader:
        rgb, mask = rgb.to(device), mask.to(device)
        optimizer.zero_grad()
        out = model(rgb)
        loss = criterion(out["out"], mask)
        if "aux" in out and out["aux"] is not None:
            loss = loss + 0.4 * criterion(out["aux"], mask)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / max(len(train_loader), 1)

    mean_iou, iou_per_class = evaluate(model, val_loader, NUM_CLASSES)
    history["train_loss"].append(avg_loss)
    history["val_mean_iou"].append(mean_iou)

    print(f"\nEpoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}  val_mean_iou={mean_iou:.4f}  ({time.time()-t0:.0f}s)")
    print_iou_table(iou_per_class)

    if mean_iou > best_iou:
        best_iou = mean_iou
        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "best_iou": best_iou,
        }, OUTPUT_DIR / CHECKPOINT_NAME)
        print(f"  -> new best (mean_iou={best_iou:.4f}), saved to {OUTPUT_DIR / CHECKPOINT_NAME}")

print(f"\nTraining done. Best val mean IoU: {best_iou:.4f}")


## 8. Visual check -- SCREENSHOT THIS for your agent

RGB | ground truth | prediction, for a handful of validation samples, plus the IoU table above. This combination (numbers + pictures) is what the agent workflow in `AGENT_FINETUNE_GUIDE.md` needs to actually diagnose what to change next -- text metrics alone hide *where* it's wrong (e.g. small buildings vs large ones, tree edges vs tree interiors).

In [ ]:
# reload the BEST checkpoint (not just whatever's in memory from the last epoch)
best_ckpt = torch.load(OUTPUT_DIR / CHECKPOINT_NAME, map_location=device)
model.load_state_dict(best_ckpt["model_state"])
model.eval()

CLASS_COLORS = np.array([
    [0, 0, 0], [139, 90, 43], [124, 191, 92], [214, 39, 40],
    [31, 119, 180], [127, 127, 127], [44, 160, 44],
], dtype=np.uint8)

def colorize(mask):
    return CLASS_COLORS[mask]

n_show = 4
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
with torch.no_grad():
    for i in range(n_show):
        rgb, mask = val_dataset[i]
        pred = model(rgb.unsqueeze(0).to(device))["out"].argmax(1).squeeze(0).cpu().numpy()
        rgb_np = (rgb.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

        axes[i, 0].imshow(rgb_np); axes[i, 0].set_title("RGB input"); axes[i, 0].axis("off")
        axes[i, 1].imshow(colorize(mask.numpy())); axes[i, 1].set_title("Ground truth"); axes[i, 1].axis("off")
        axes[i, 2].imshow(colorize(pred)); axes[i, 2].set_title("Prediction"); axes[i, 2].axis("off")

plt.suptitle(f"Run: {RUN_NAME}  |  best_val_mean_iou={best_ckpt['best_iou']:.4f}", y=1.0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"visual_check_{RUN_NAME}.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR / f'visual_check_{RUN_NAME}.png'} -- screenshot this cell's output (or grab that file) to share.")


## 9. Loss / IoU curves (screenshot this too)

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(history["train_loss"], label="train loss", color="tab:red")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss", color="tab:red")
ax2 = ax1.twinx()
ax2.plot(history["val_mean_iou"], label="val mean IoU", color="tab:blue")
ax2.set_ylabel("val mean IoU", color="tab:blue")
plt.title(f"Run: {RUN_NAME}")
fig.tight_layout()
plt.savefig(OUTPUT_DIR / f"curves_{RUN_NAME}.png", dpi=100)
plt.show()


## 10. Export

Two options -- pick whichever you have set up:


In [ ]:
# Option A: mount Google Drive and copy the checkpoint + screenshots there
from google.colab import drive
drive.mount("/content/drive")

import shutil
export_dir = Path(f"/content/drive/MyDrive/depthwizard_checkpoints/{RUN_NAME}")
export_dir.mkdir(parents=True, exist_ok=True)
for f in OUTPUT_DIR.glob("*"):
    shutil.copy(f, export_dir / f.name)
print(f"Exported to Google Drive: {export_dir}")


In [ ]:
# Option B: direct download to your machine instead of Drive
from google.colab import files
files.download(str(OUTPUT_DIR / CHECKPOINT_NAME))


## 11. What to try next (this is what the agent loop is for)

Things worth experimenting with across runs -- change ONE at a time so you can tell what actually helped:

- `LEARNING_RATE` -- if val IoU is noisy/unstable, try lower (1e-4); if it's barely moving, try higher
- `USE_CLASS_WEIGHTS` -- on/off, and try manually overriding specific class weights if one class's IoU is consistently the worst
- `AUGMENT` -- on/off, or add more augmentations (rotation, more aggressive color jitter) directly in `GAMUSSegDataset`
- `FREEZE_BACKBONE_EPOCHS` -- freezing the backbone for a few epochs first can stabilize fine-tuning from the existing checkpoint
- `NUM_TRAIN_SAMPLES` -- more data if you have Colab session time to spare
- Loss function -- current is weighted cross-entropy; a combined CE + Dice loss often helps IoU on rare/small classes (building, water) specifically, worth trying as its own run

Every run should get its own `RUN_NAME` so nothing overwrites another attempt.
Do NOT rename any output file to `landcover_seg_v6.pth` yourselves -- once
you have a checkpoint that's genuinely better, hand it back with its IoU
table and screenshots and that swap gets made deliberately, not silently.
